[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/dlt-certified/blob/main/notebooks/day-02-sources.ipynb)

---
# Day 2 · Sources, Resources, and Decorators
**certified-journeys / dlt-certified** &nbsp;|&nbsp; Data Ingestion

> **Goal for today:** Master the `@dlt.source` and `@dlt.resource` decorators, build a custom REST API source using dlt's `rest_api` helper, configure pagination and authentication, and inspect the generated schema.

---
## The anatomy of a dlt source

In Day 1 you saw the simplest possible resource — a generator wrapped in `@dlt.resource`. Today we go deeper. Real pipelines need:

- Multiple related resources grouped under one **source**
- Resources that **feed each other** (transformers)
- Resources that call **REST APIs** with pagination and auth
- Fine-grained control over **how** data lands (name, primary key, write disposition)

Here is the full decorator option surface you'll use today:

| Option | What it controls | Example value |
|---|---|---|
| `name` | Table name in the destination | `"posts"` |
| `primary_key` | Column(s) used for deduplication with `merge` | `"id"` or `["id", "type"]` |
| `write_disposition` | How rows land: append / replace / merge | `"replace"` |
| `selected` | Whether this resource runs by default | `True` (default) |
| `parallelized` | Run resource concurrently with others | `True` |
| `table_name` | Dynamic table name via callable | `lambda row: row["type"]` |

---
## How `@dlt.source` and `@dlt.resource` relate

```
@dlt.source
def my_api():
    @dlt.resource          # resource A
    def users(): ...

    @dlt.resource          # resource B
    def posts(): ...

    return users, posts    # ← the source exposes both
```

When you call `pipeline.run(my_api())` dlt runs all returned resources. You can also run just one: `pipeline.run(my_api().posts)`.

### Transformer resources

A **transformer** is a resource that takes rows from another resource and enriches them. The `@dlt.transformer` decorator (or `pipe_from`) wires the dependency:

```
users ──▶ user_details   (one API call per user row)
```

This is the pattern used when you need to paginate through IDs fetched from an index endpoint.

---
## Step 1 · Install dlt

In [ ]:
%pip install -q "dlt[duckdb]"

---
## Step 2 · `@dlt.resource` decorator options in depth

Let's explore every important decorator option before we connect to a real API.

In [ ]:
import dlt

# name=       → controls the table name (default: function name)
# primary_key= → which column uniquely identifies a row
# write_disposition= → how rows land in the destination
# selected=   → False means the resource exists but won't run unless explicitly asked

@dlt.resource(
    name="employees",           # table will be called 'employees', not 'staff'
    primary_key="emp_id",       # used when write_disposition='merge'
    write_disposition="replace", # wipe table, then insert
    selected=True,              # included in source runs by default
)
def staff():
    """Yields employee records."""
    yield [
        {"emp_id": 1, "name": "Alice",   "dept": "Engineering"},
        {"emp_id": 2, "name": "Bob",     "dept": "Analytics"},
        {"emp_id": 3, "name": "Charlie", "dept": "Engineering"},
    ]


# A second resource that is NOT selected by default
@dlt.resource(
    name="departments",
    write_disposition="replace",
    selected=False,   # ← won't run unless explicitly included
)
def dept_lookup():
    yield [
        {"dept": "Engineering", "head_count_limit": 50},
        {"dept": "Analytics",   "head_count_limit": 20},
    ]


pipeline = dlt.pipeline(
    pipeline_name="day02_demo",
    destination="duckdb",
    dataset_name="hr",
)

# Run only the selected resource (staff)
info = pipeline.run(staff())
print(info)

with pipeline.sql_client() as client:
    with client.execute_query("SELECT * FROM hr.employees") as cur:
        print(cur.df())

**What just happened?**
- The function is named `staff` but the table is named `employees` because we set `name="employees"`.
- `write_disposition="replace"` means the whole table is truncated and reloaded on every run.
- `dept_lookup` was defined but not run — `selected=False` keeps it dormant until explicitly invoked.

Now run both resources explicitly:

In [ ]:
# Force-include the non-selected resource by passing it directly
info2 = pipeline.run([staff(), dept_lookup()])
print(info2)

with pipeline.sql_client() as client:
    for tbl in ["hr.employees", "hr.departments"]:
        print(f"\n── {tbl} ──")
        with client.execute_query(f"SELECT * FROM {tbl}") as cur:
            print(cur.df())

---
## Step 3 · Grouping resources with `@dlt.source`

The `@dlt.source` decorator turns a function that returns resources into a **single callable unit**. This is the standard pattern for any multi-table API wrapper.

In [ ]:
@dlt.source(name="crm")
def crm_source(region: str = "us"):
    """
    A source that groups two related resources.
    The 'region' parameter is passed through to both resources —
    this is the dlt pattern for per-source configuration.
    """

    @dlt.resource(primary_key="id", write_disposition="replace")
    def contacts():
        # In real life: call your CRM API here
        yield [
            {"id": 1, "email": "alice@example.com", "region": region, "stage": "lead"},
            {"id": 2, "email": "bob@example.com",   "region": region, "stage": "customer"},
        ]

    @dlt.resource(primary_key="id", write_disposition="replace")
    def deals():
        yield [
            {"id": 101, "contact_id": 1, "value": 5000,  "status": "open",   "region": region},
            {"id": 102, "contact_id": 2, "value": 12000, "status": "closed", "region": region},
        ]

    return contacts, deals


pipeline_crm = dlt.pipeline(
    pipeline_name="crm_pipeline",
    destination="duckdb",
    dataset_name="crm",
)

# Run the whole source
info3 = pipeline_crm.run(crm_source(region="eu"))
print(info3)

with pipeline_crm.sql_client() as client:
    for tbl in ["crm.contacts", "crm.deals"]:
        print(f"\n── {tbl} ──")
        with client.execute_query(f"SELECT * FROM {tbl}") as cur:
            print(cur.df())

**What just happened?**
- `crm_source(region="eu")` passes the `region` argument into both nested resources.
- You can run a single resource from the source: `pipeline_crm.run(crm_source().contacts)` — no changes needed to the source itself.
- dlt created two tables: `crm.contacts` and `crm.deals`.

---
## Step 4 · Transformer resources (one resource feeds another)

A transformer takes each row from a parent resource and yields new rows — perfect for detail endpoints that require an ID from an index endpoint.

In [ ]:
# Parent resource: yields user IDs (cheap index endpoint)
@dlt.resource(primary_key="id", write_disposition="replace")
def user_ids():
    yield from [{"id": i} for i in range(1, 4)]  # users 1, 2, 3


# Transformer: takes each row from user_ids and enriches it
# data_from=  wires the parent; the function receives parent rows one at a time
@dlt.transformer(
    data_from=user_ids,
    primary_key="id",
    write_disposition="replace",
)
def user_details(user_id_row):
    """Simulates fetching /users/{id} for each user ID."""
    uid = user_id_row["id"]
    # In real life: response = requests.get(f"/users/{uid}").json()
    fake_details = {
        1: {"id": 1, "name": "Alice",   "email": "alice@example.com", "score": 92},
        2: {"id": 2, "name": "Bob",     "email": "bob@example.com",   "score": 78},
        3: {"id": 3, "name": "Charlie", "email": "charlie@example.com","score": 85},
    }
    yield fake_details[uid]


pipeline_t = dlt.pipeline(
    pipeline_name="transformer_demo",
    destination="duckdb",
    dataset_name="users",
)

# Run only the transformer — dlt automatically pulls from user_ids first
info4 = pipeline_t.run(user_details)
print(info4)

with pipeline_t.sql_client() as client:
    with client.execute_query("SELECT * FROM users.user_details") as cur:
        print(cur.df())

**What just happened?**
- `user_ids` runs first and yields rows `{"id": 1}`, `{"id": 2}`, `{"id": 3}`.
- For each row, `user_details` is called and yields the enriched record.
- dlt only stores the `user_details` table (the output). The `user_ids` intermediate resource is not persisted unless you explicitly include it.

---
## Step 5 · Real API calls with the `rest_api` helper

dlt ships a powerful `rest_api` source factory that handles:
- Pagination (cursor, page number, link header, offset)
- Authentication (Bearer token, API key, OAuth)
- Rate limiting
- Multiple endpoints in one config block

We'll use [JSONPlaceholder](https://jsonplaceholder.typicode.com) — a free, public, no-auth REST API — to keep this runnable without credentials.

In [ ]:
from dlt.sources.rest_api import rest_api_source

# The rest_api_source() factory accepts a single config dict.
# It returns a dlt source with one resource per endpoint defined.

jsonplaceholder = rest_api_source(
    {
        # ── Client config: applies to every endpoint ──────────────────────
        "client": {
            "base_url": "https://jsonplaceholder.typicode.com",
            # No auth needed for this API.
            # For a Bearer token you'd add:
            # "auth": {"type": "bearer", "token": dlt.secrets["api_token"]}
        },

        # ── Default settings shared across all resources ───────────────────
        "resource_defaults": {
            "primary_key": "id",
            "write_disposition": "replace",
        },

        # ── Per-endpoint resource definitions ─────────────────────────────
        "resources": [
            {
                "name": "posts",
                "endpoint": {
                    "path": "/posts",
                    # JSONPlaceholder returns all 100 posts in one response
                    # (no real pagination needed), but we configure
                    # PageNumberPaginator to show how it works.
                    "paginator": {
                        "type": "page_number",
                        "base_page": 1,
                        "page": "_page",       # query param name
                        "total_path": None,    # unknown total — stop on empty response
                        "maximum_page": 1,     # fetch only page 1 for this demo
                    },
                },
            },
            {
                "name": "users",
                "endpoint": {
                    "path": "/users",
                    # This endpoint returns all users in a single response
                    "paginator": "single_page",
                },
            },
        ],
    }
)

pipeline_api = dlt.pipeline(
    pipeline_name="jsonplaceholder",
    destination="duckdb",
    dataset_name="jp",
)

info5 = pipeline_api.run(jsonplaceholder)
print(info5)

**What just happened?**
- `rest_api_source()` handled the HTTP calls, response parsing, and pagination automatically.
- Two tables were created: `jp.posts` (100 rows) and `jp.users` (10 rows).
- `primary_key="id"` and `write_disposition="replace"` came from `resource_defaults` and applied to both endpoints.

Inspect the data:

In [ ]:
with pipeline_api.sql_client() as client:
    print("── posts (first 5) ──")
    with client.execute_query("SELECT id, user_id, title FROM jp.posts LIMIT 5") as cur:
        print(cur.df())

    print("\n── users ──")
    with client.execute_query("SELECT id, name, email, username FROM jp.users") as cur:
        print(cur.df())

    print("\n── post count ──")
    with client.execute_query("SELECT COUNT(*) AS total FROM jp.posts") as cur:
        print(cur.df())

---
## Step 6 · Inspect the generated schema

dlt inferred column types from the JSON responses. Let's examine what it built.

In [ ]:
# The default schema contains all tables loaded in this pipeline
schema = pipeline_api.default_schema

# Print as YAML — human-readable, shows every column and its inferred type
print(schema.to_pretty_yaml())

In [ ]:
# You can also inspect individual table schemas programmatically
posts_schema = schema.get_table("posts")
print("posts columns:")
for col_name, col_def in posts_schema["columns"].items():
    dtype = col_def.get("data_type", "(not set)")
    pk    = " [PK]" if col_def.get("primary_key") else ""
    print(f"  {col_name:<25} {dtype}{pk}")

---
## Step 7 · Authentication patterns (reference — no live call needed)

You won't need auth for the challenge, but here are the three most common patterns for real APIs:

```python
# Bearer token (most common)
"client": {
    "base_url": "https://api.example.com",
    "auth": {
        "type": "bearer",
        "token": dlt.secrets["sources.my_api.token"],
    }
}

# API key in header
"client": {
    "base_url": "https://api.example.com",
    "auth": {
        "type": "api_key",
        "name": "X-Api-Key",
        "api_key": dlt.secrets["sources.my_api.api_key"],
        "location": "header",
    }
}

# API key as query parameter
"client": {
    "base_url": "https://api.example.com",
    "auth": {
        "type": "api_key",
        "name": "apikey",
        "api_key": dlt.secrets["sources.my_api.api_key"],
        "location": "query",
    }
}
```

Store secrets in `.dlt/secrets.toml` (never in source code):
```toml
[sources.my_api]
token = "your-token-here"
```

---
## Challenge — do this yourself

Use `rest_api_source` to fetch the `/comments` and `/albums` endpoints from JSONPlaceholder.

Requirements:
1. Both resources should use `primary_key="id"` and `write_disposition="replace"`.
2. Load into a new pipeline named `"jp_challenge"` with `dataset_name="challenge"`.
3. After loading, print the row count for each table.
4. Bonus: print the column names of the `comments` table from the schema.

**Hint:** `/comments` returns 500 rows, `/albums` returns 100 rows.

In [ ]:
# Your solution here


<details>
<summary>Show solution</summary>

```python
from dlt.sources.rest_api import rest_api_source
import dlt

source = rest_api_source(
    {
        "client": {"base_url": "https://jsonplaceholder.typicode.com"},
        "resource_defaults": {
            "primary_key": "id",
            "write_disposition": "replace",
        },
        "resources": [
            {"name": "comments", "endpoint": {"path": "/comments", "paginator": "single_page"}},
            {"name": "albums",   "endpoint": {"path": "/albums",   "paginator": "single_page"}},
        ],
    }
)

p = dlt.pipeline(pipeline_name="jp_challenge", destination="duckdb", dataset_name="challenge")
p.run(source)

with p.sql_client() as client:
    for tbl in ["challenge.comments", "challenge.albums"]:
        with client.execute_query(f"SELECT COUNT(*) AS total FROM {tbl}") as cur:
            print(f"{tbl}: {cur.df()['total'][0]} rows")

# Bonus: column names from schema
schema = p.default_schema
comments_cols = list(schema.get_table("comments")["columns"].keys())
print("comments columns:", comments_cols)
```
</details>

---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| `@dlt.resource(name=...)` | Overrides the table name — decouple function name from destination table |
| `@dlt.resource(selected=False)` | Resource is defined but opt-in — useful for expensive or rarely-needed tables |
| `@dlt.source` | Groups resources under one callable; accepts config parameters |
| `@dlt.transformer` | A resource that takes rows from another resource as input |
| `rest_api_source()` | Config-driven factory: base URL + per-endpoint config = full source |
| `paginator: "single_page"` | Tells dlt the endpoint returns everything in one response |
| `paginator: "page_number"` | dlt increments a `_page` query param and stops on empty response |
| Schema inspection | `pipeline.default_schema.to_pretty_yaml()` shows all inferred types |

> **Tip:** Use dlt's [verified sources](https://github.com/dlt-hub/verified-sources) as reference implementations — they show all best practices including incremental loading, auth, and pagination.

---
## What's next → Day 3

**Day 3** → Destinations and loading strategies: `append`, `replace`, `merge`, composite primary keys, and inspecting load traces.

Mark Day 2 complete in your [tracker](../index.html).